# Big West championship extraction — expanded compact field-event summaries

Uses compact official field-event results (PL/NAME/YEAR/TEAM/MARK/SC, optional CONV/WIND/ENGLISH), ignoring attempt and flight tables. Running-event parsing remains provisional.

In [1]:
from pathlib import Path
from io import StringIO
import re
import pandas as pd
from bs4 import BeautifulSoup

source_path = Path('../data/raw/bigwest_2026_women.html')
output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

soup = BeautifulSoup(source_path.read_text(encoding='utf-8'), 'html.parser')
for tag in soup.find_all(attrs={'colspan': True}):
    if not str(tag['colspan']).isdigit():
        tag['colspan'] = '1'

table_tags = soup.find_all('table')
tables = pd.read_html(StringIO(str(soup)))
assert len(table_tags) == len(tables), 'HTML/Pandas table counts differ; check table alignment'
print('Tables found:', len(tables))


c:\Users\12094\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Tables found: 69


In [2]:
FIELD_KEYWORDS = ('long jump', 'triple jump', 'high jump', 'pole vault',
                  'shot put', 'discus', 'hammer', 'javelin')
COMBINED_KEYWORDS = ('heptathlon', 'decathlon', 'pentathlon')

def get_event_info(table_tag):
    heading = table_tag.find_previous(['h1', 'h2', 'h3', 'h4', 'h5'])
    while heading:
        text = heading.get_text(' ', strip=True)
        if re.search(r"(?:Women['’]s|Men['’]s)", text, re.I):
            break
        heading = heading.find_previous(['h1', 'h2', 'h3', 'h4', 'h5'])
    if not heading:
        return None, None, None, None
    text = heading.get_text(' ', strip=True)
    event = re.split(r'\b(?:Finals?|Preliminaries|Semi-Finals?|Flight|Heat|Section)\b',
                     text, maxsplit=1, flags=re.I)[0]
    event = re.sub(r"^(?:Women['’]s|Men['’]s)\s+", '', event).strip()
    low = event.lower()
    event_type = ('combined' if any(x in low for x in COMBINED_KEYWORDS)
                  else 'field' if any(x in low for x in FIELD_KEYWORDS)
                  else 'running')
    round_name = ('Flight' if re.search(r'\bFlight\b', text, re.I)
                  else 'Heat' if re.search(r'\bHeat\b', text, re.I)
                  else 'Preliminaries' if re.search(r'\bPreliminaries\b', text, re.I)
                  else 'Finals' if re.search(r'\bFinals?\b', text, re.I)
                  else 'Other')
    return event, event_type, round_name, text

def get_official_mark_column(df, event_type, round_name):
    if event_type == 'field':
        # Field events must use the official main results table, never MARK.1 etc.
        return 'MARK' if is_field_summary_table(df) else None
    cols = [c for c in df.columns if re.fullmatch(r'TIME(?:\.\d+)?', str(c))]
    if not cols:
        return None
    # Running-time selection is provisional and should be independently checked.
    return cols[0] if round_name in ('Flight', 'Heat', 'Preliminaries') else cols[-1]

def is_field_summary_table(df):
    """Accept compact official results, regardless of optional CONV/WIND fields.

    Reject tables with MARK.1, MARK.2, attempts, or other expanded columns.
    """
    cols = [str(c).strip().upper() for c in df.columns]
    required = {'PL', 'NAME', 'YEAR', 'TEAM', 'MARK', 'SC'}
    allowed = required | {'CONV', 'WIND', 'ENGLISH'}
    return (required.issubset(cols)
            and set(cols).issubset(allowed)
            and len(cols) == len(set(cols)))


In [3]:
def clean_event_table(table, tag, index):
    event, event_type, round_name, heading = get_event_info(tag)
    if not event or event_type == 'combined':
        return None
    df = table.copy()
    if event_type == 'field':
        # ONLY compact main field results, with optional CONV/WIND/ENGLISH.
        # No flight results, detailed attempts, or extra MARK columns.
        if round_name in ('Flight', 'Heat', 'Preliminaries') or re.search(r'\b(?:Flight|Heat|Preliminaries)\b', heading, re.I):
            return None
        if not is_field_summary_table(df):
            return None
    elif not {'PL', 'NAME', 'TEAM'}.issubset(df.columns):
        return None

    mark_col = get_official_mark_column(df, event_type, round_name)
    if mark_col is None:
        return None
    df = df.rename(columns={'PL':'place', 'NAME':'athlete_name', 'YEAR':'class_year',
                            'TEAM':'school', 'SC':'points', 'WIND':'wind',
                            'CONV':'converted_mark', 'ENGLISH':'english_mark', mark_col:'mark'})
    ids = []
    for tr in tag.select('tbody tr'):
        link = tr.select_one('a[href*="/athletes/"]')
        m = re.search(r'/athletes/(\d+)', link.get('href', '')) if link else None
        ids.append(m.group(1) if m else None)
    if len(ids) != len(df):
        print(f'SKIP table {index}: athlete-link rows {len(ids)} != parsed rows {len(df)}')
        return None
    df['athlete_id'] = ids
    df = df[df['athlete_id'].notna() & df['athlete_name'].notna() & df['school'].notna()].copy()
    if df.empty:
        return None
    for optional in ('class_year', 'wind', 'converted_mark', 'english_mark'):
        if optional not in df: df[optional] = None
    if 'points' not in df: df['points'] = 0
    df['points'] = pd.to_numeric(df['points'], errors='coerce').fillna(0)
    df['place'] = pd.to_numeric(df['place'], errors='coerce')
    df['year'], df['gender'], df['event'] = 2026, 'Women', event
    df['event_type'], df['round'] = event_type, round_name
    df['source_table'], df['source_heading'] = index, heading
    df['mark_source_column'] = mark_col
    return df[['year','gender','event','event_type','round','athlete_id',
               'athlete_name','class_year','school','place','mark','converted_mark',
               'points','wind','english_mark','source_table','source_heading','mark_source_column']]

parsed = []
field_candidates = set()
field_selected = set()
for i, (table, tag) in enumerate(zip(tables, table_tags)):
    event, event_type, round_name, heading = get_event_info(tag)
    if event_type == 'field':
        field_candidates.add(event)
    try:
        result = clean_event_table(table, tag, i)
        if result is not None:
            parsed.append(result)
            if event_type == 'field':
                field_selected.add(event)
                print(f'FIELD SUMMARY: table {i}, {event}, {len(result)} athletes')
    except Exception as exc:
        print(f'Error on table {i}: {exc}')

missing = sorted(field_candidates - field_selected)
if missing:
    print('WARNING: no matching compact main results table for:', missing)
    print('These events are OMITTED rather than using unreliable attempt/flight marks.')
if not parsed:
    raise ValueError('No result tables parsed. Inspect source HTML and table columns.')
all_results = pd.concat(parsed, ignore_index=True)
print('Parsed rows:', len(all_results))


FIELD SUMMARY: table 59, Discus, 24 athletes
FIELD SUMMARY: table 62, Hammer, 24 athletes
These events are OMITTED rather than using unreliable attempt/flight marks.
Parsed rows: 572


In [4]:
# Field rows come ONLY from compact official summary tables.
# Running results still prioritize finals over prelims and retain prelim-only athletes.
priority = {'Finals': 4, 'Other': 3, 'Preliminaries': 2, 'Flight': 1, 'Heat': 1}
all_results['round_priority'] = all_results['round'].map(priority).fillna(0)
all_results['has_mark'] = all_results['mark'].notna() & ~all_results['mark'].astype(str).str.strip().eq('')
all_results = all_results.sort_values(
    ['round_priority', 'has_mark', 'points', 'source_table'],
    ascending=[False, False, False, True]
)
selected = all_results.drop_duplicates(subset=['year','gender','event','athlete_id'], keep='first').copy()
running_df = selected[selected['event_type']=='running'].copy()
field_df = selected[selected['event_type']=='field'].copy()
assert (field_df['mark_source_column'] == 'MARK').all(), 'Unexpected field mark column'
assert field_df['source_table'].map(lambda i: is_field_summary_table(tables[int(i)])).all(), 'Non-summary field table used'
print('Running rows:', len(running_df), '| Field rows:', len(field_df))


Running rows: 269 | Field rows: 48


In [5]:
# Check source tables, results and totals for each field event.
print('FIELD RESULTS BY EVENT (including missing-event warnings above)')
display(field_df.groupby('event').agg(entries=('athlete_name','size'),
                                     points=('points','sum'),
                                     source_tables=('source_table', lambda s: sorted(s.unique()))))
for event in ['Triple Jump', 'Javelin', 'Discus']:
    subset = field_df[field_df['event'].str.fullmatch(event, case=False, na=False)]
    print(f'\n{event}: {len(subset)} entries')
    display(subset[['athlete_name','place','mark','converted_mark','points','source_table']].sort_values('place'))
print('\nRUNNING EVENTS')
display(running_df.groupby('event').agg(entries=('athlete_name','size'), points=('points','sum')))


FIELD RESULTS BY EVENT (including missing-event warnings above)


,entries,points,source_tables
event,,,
Discus,24,39.0,[59]
Hammer,24,39.0,[62]



Triple Jump: 0 entries


,athlete_name,place,mark,converted_mark,points,source_table



Javelin: 0 entries


,athlete_name,place,mark,converted_mark,points,source_table



Discus: 24 entries


,athlete_name,place,mark,converted_mark,points,source_table
524,Summer Mosley,1.0,59.60m,"195' 6""",10.0,59
525,Katherine Shipp-Davis,2.0,55.28m,"181' 4""",8.0,59
526,Nyla Baker,3.0,55.01m,"180' 6""",6.0,59
527,Samaria Vital,4.0,51.95m,"170' 5""",5.0,59
528,Amani Britton,5.0,51.04m,"167' 5""",4.0,59
529,Rose Forshaw,6.0,48.51m,"159' 2""",3.0,59
530,Dimitra Vitogiannis,7.0,48.44m,"158' 11""",2.0,59
531,Feyi Olukanni,8.0,48.20m,"158' 2""",1.0,59
532,Kayla Yap,9.0,47.98m,"157' 5""",0.0,59
533,Camryn Alo,10.0,47.64m,"156' 3""",0.0,59



RUNNING EVENTS


,entries,points
event,,
"10,000 Meters",23,39.0
100 Hurdles,19,39.0
100 Meters,36,39.0
1500 Meters,23,39.0
200 Meters,34,39.0
3000 Steeplechase,18,39.0
400 Hurdles,22,39.0
400 Meters,29,39.0
5000 Meters,38,39.0


In [6]:
# Verify coverage of all eight individual field disciplines.
EXPECTED_FIELD_EVENTS = ['High Jump', 'Pole Vault', 'Long Jump', 'Triple Jump',
                         'Shot Put', 'Discus', 'Hammer', 'Javelin']
print('Field events extracted:', sorted(field_df['event'].unique()))
for event in EXPECTED_FIELD_EVENTS:
    matched = field_df[field_df['event'].str.contains(event, case=False, regex=False, na=False)]
    print(f'{event:15} {len(matched):3} entries')
    if not matched.empty:
        display(matched[['event','athlete_name','place','mark','points','source_table']].head(4))
print('NOTE: Any missing events require inspecting their actual summary-table columns.')

Field events extracted: ['Discus', 'Hammer']
High Jump         0 entries
Pole Vault        0 entries
Long Jump         0 entries
Triple Jump       0 entries
Shot Put          0 entries
Discus           24 entries


,event,athlete_name,place,mark,points,source_table
524,Discus,Summer Mosley,1.0,59.60m,10.0,59
525,Discus,Katherine Shipp-Davis,2.0,55.28m,8.0,59
526,Discus,Nyla Baker,3.0,55.01m,6.0,59
527,Discus,Samaria Vital,4.0,51.95m,5.0,59


Hammer           24 entries


,event,athlete_name,place,mark,points,source_table
548,Hammer,Jocelyn Alexander,1.0,57.38m,10.0,62
549,Hammer,Giszelle Hrehor,2.0,57.36m,8.0,62
550,Hammer,Deana Davis,3.0,56.36m,6.0,62
551,Hammer,Amanda Javellana,4.0,55.91m,5.0,62


Javelin           0 entries
NOTE: Any missing events require inspecting their actual summary-table columns.


In [ ]:
# Export only after reviewing the diagnostics above.
running_path = output_dir / 'bigwest_2026_women_running.csv'
field_path = output_dir / 'bigwest_2026_women_field.csv'
running_df.to_csv(running_path, index=False)
field_df.to_csv(field_path, index=False)
print('Saved:', running_path, field_path, sep='\n')


Saved:
..\data\processed\bigwest_2026_women_running.csv
..\data\processed\bigwest_2026_women_field.csv


: 

## Quality notes
- Field extraction **requires exactly** `PL, NAME, YEAR, TEAM, MARK, CONV, SC`, and uses `MARK` only.
- Flight tables and detailed attempt tables are ignored. Events without the expected summary table are reported and omitted (no guessed marks).
- The exact seven-column schema should be checked against your saved HTML. Running `TIME` selection remains a heuristic; audit it separately.
- Athletes with no valid mark (FOUL, NM, DNS) can remain when their profile row exists.
- Relays and combined-event subevents may need separate extraction.
